# Week 5 — Kaggle Experiments

**One notebook, four Kaggle accounts. Only `EXPERIMENT` changes per account.**

| Account | EXPERIMENT | Est. time |
|---------|------------|-----------|
| 1 | `E1_classical` | ~7.5 hrs |
| 2 | `E3a_classical_20pct` | ~1.5 hrs |
| 3 | `E3b_quantum_20pct` | ~7 hrs |
| 4 | `E4_resnet` | ~3.5 hrs |

E2 (quantum full dataset) runs locally — too long for Kaggle 12-hr limit.

**Required Kaggle datasets attached to this notebook:**
- `quantapath-code` — project source code (pathq/ modules)
- `quantapath-features` — UNI 1024-dim .pt feature files
- `quantapath-resnet-features` — ResNet-50 512-dim .pt feature files (E4 only)

In [1]:
# ══════════════════════════════════════════════════════════
#  CHANGE THIS ONE LINE PER KAGGLE ACCOUNT:
#
#  Account 1  →  EXPERIMENT = 'E1_classical'
#  Account 2  →  EXPERIMENT = 'E3a_classical_20pct'
#  Account 3  →  EXPERIMENT = 'E3b_quantum_20pct'
#  Account 4  →  EXPERIMENT = 'E4_resnet'
# ══════════════════════════════════════════════════════════
EXPERIMENT = 'E1_classical'

# What each experiment does:
# E1_classical        → Full dataset,  no VQC, 50 epochs  → Paper Table 1 Row 1
# E3a_classical_20pct → 20% data,      no VQC, 30 epochs  → Paper Table 2 Row 1
# E3b_quantum_20pct   → 20% data,      VQC ON, 30 epochs  → Paper Table 2 Row 2 (KEY)
# E4_resnet           → ResNet-50,     no VQC, 30 epochs  → Paper Table 1 Row 3

print(f'Running experiment: {EXPERIMENT}')

Running experiment: E1_classical


In [2]:
import subprocess, sys

print('Installing packages... (~3 min)')
packages = [
    'pennylane',
    'pennylane-lightning',
    'torch-geometric',
    'timm',
    'scikit-learn',
]
for pkg in packages:
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', pkg],
        check=False
    )
print('All packages installed.')

Installing packages... (~3 min)
All packages installed.


In [3]:
import os, sys, time, json, gc
import numpy as np
import torch
import torch.nn.functional as F
from pathlib import Path
from sklearn.metrics import roc_auc_score, f1_score
from torch.utils.data import Subset
from torch_geometric.loader import DataLoader as PyGLoader

# Prevent CUDA OOM fragmentation
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = \
    'expandable_segments:True,max_split_size_mb:512'

# ── Paths ──────────────────────────────────────────────────────
CODE_DIR    = Path('/kaggle/input/quantapath-code')
UNI_DIR     = Path('/kaggle/input/quantapath-features')
RESNET_DIR  = Path('/kaggle/input/quantapath-resnet-features')
CKPT_DIR    = Path('/kaggle/working/checkpoints')
OUT_DIR     = Path('/kaggle/working/outputs')

CKPT_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True,  exist_ok=True)

# ── Add project code to path ─────────────────────────────────
sys.path.insert(0, str(CODE_DIR))
from pathq.model_v2   import QuantaPathV2
from pathq.dataset_v2 import get_loaders_from_features

# ── Device ─────────────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(42)
np.random.seed(42)

print(f'Experiment : {EXPERIMENT}')
print(f'Device     : {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU        : {torch.cuda.get_device_name(0)}')
    print(f'VRAM       : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
print(f'UNI slides : {len(list(UNI_DIR.glob("*.pt")))}')

/home/kabi/.conda/envs/pathq/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PermissionError: [Errno 13] Permission denied: '/kaggle'

In [4]:
# ── Experiment configs ──────────────────────────────────────────────
CONFIGS = {
    'E1_classical': {
        'feat_dir'   : UNI_DIR,
        'use_vqc'    : False,
        'n_qubits'   : 3,
        'n_layers'   : 2,
        'in_dim'     : 1040,    # UNI 1024 + pos_enc 16
        'epochs'     : 50,
        'lr'         : 1e-4,
        'patience'   : 7,
        'low_data'   : False,
        'description': 'Full dataset | Classical GAT-Transformer | UNI features',
    },
    'E3a_classical_20pct': {
        'feat_dir'   : UNI_DIR,
        'use_vqc'    : False,
        'n_qubits'   : 3,
        'n_layers'   : 2,
        'in_dim'     : 1040,
        'epochs'     : 30,
        'lr'         : 1e-4,
        'patience'   : 7,
        'low_data'   : True,
        'description': '20% data | Classical GAT-Transformer | UNI features',
    },
    'E3b_quantum_20pct': {
        'feat_dir'   : UNI_DIR,
        'use_vqc'    : True,
        'n_qubits'   : 3,       # Week 4 winner
        'n_layers'   : 2,       # Week 4 winner
        'in_dim'     : 1040,
        'epochs'     : 30,
        'lr'         : 3e-5,    # slower lr for VQC stability
        'patience'   : 5,
        'low_data'   : True,
        'description': '20% data | Quantum VQC+GAT (3q 2L) | UNI features',
    },
    'E4_resnet': {
        'feat_dir'   : RESNET_DIR,
        'use_vqc'    : False,
        'n_qubits'   : 3,
        'n_layers'   : 2,
        'in_dim'     : 528,     # ResNet-50 512 + pos_enc 16
        'epochs'     : 30,
        'lr'         : 1e-4,
        'patience'   : 7,
        'low_data'   : False,
        'description': 'Full dataset | Classical GAT-Transformer | ResNet-50 features',
    },
}

cfg = CONFIGS[EXPERIMENT]
print(f'Description : {cfg["description"]}')
print(f'use_vqc     : {cfg["use_vqc"]}')
print(f'in_dim      : {cfg["in_dim"]}')
print(f'epochs      : {cfg["epochs"]}')
print(f'lr          : {cfg["lr"]}')
print()

# ── Load full loaders ──────────────────────────────────────────────
MAX_PATCHES = 3000  # paper-quality numbers

train_loader, val_loader, test_loader = get_loaders_from_features(
    features_dir = cfg['feat_dir'],
    batch_size   = 4,
    k            = 8,
    seed         = 42,
    max_patches  = MAX_PATCHES,
)

# ── Subset to 20% for low-data experiments ─────────────────────
if cfg['low_data']:
    train_ds   = train_loader.dataset
    n_total    = len(train_ds)
    all_labels = [train_ds.get(i).y.item() for i in range(n_total)]

    pos_idx = [i for i, l in enumerate(all_labels) if l == 1]
    neg_idx = [i for i, l in enumerate(all_labels) if l == 0]

    n_half  = max(int(n_total * 0.10), 2)
    rng     = np.random.default_rng(42)
    rng.shuffle(pos_idx); rng.shuffle(neg_idx)
    subset_idx = pos_idx[:n_half] + neg_idx[:n_half]

    train_loader = PyGLoader(
        Subset(train_ds, subset_idx),
        batch_size  = 4,
        shuffle     = True,
        num_workers = 0,
    )
    n_pos = sum(all_labels[i] for i in subset_idx)
    print(f'20% subset  : {len(subset_idx)} slides '
          f'(pos={n_pos} neg={len(subset_idx)-n_pos})')

print(f'Train batches: {len(train_loader)}')
print(f'Val batches  : {len(val_loader)}')
print(f'Test batches : {len(test_loader)}')

Description : Full dataset | Classical GAT-Transformer | UNI features
use_vqc     : False
in_dim      : 1040
epochs      : 50
lr          : 0.0001



NameError: name 'get_loaders_from_features' is not defined

In [ ]:
def train_one_epoch(model, loader, optimizer, device):
    model.train()
    total_loss, n_batches = 0.0, 0
    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        torch.cuda.empty_cache()
        logits, _  = model(batch)
        loss       = F.cross_entropy(logits, batch.y.view(-1))
        loss_val   = loss.item()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        del logits, loss
        torch.cuda.empty_cache()
        total_loss += loss_val
        n_batches  += 1
    return total_loss / max(n_batches, 1)


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    all_probs, all_labels = [], []
    total_loss, n_batches = 0.0, 0
    for batch in loader:
        batch      = batch.to(device)
        logits, _  = model(batch)
        total_loss += F.cross_entropy(logits, batch.y.view(-1)).item()
        all_probs.extend(torch.softmax(logits, 1)[:, 1].cpu().tolist())
        all_labels.extend(batch.y.view(-1).cpu().tolist())
        n_batches  += 1
    probs  = np.array(all_probs)
    labels = np.array(all_labels)
    preds  = (probs >= 0.5).astype(int)
    auc    = roc_auc_score(labels, probs) if len(np.unique(labels)) > 1 else 0.5
    f1     = f1_score(labels, preds, zero_division=0)
    tp = int(((preds==1) & (labels==1)).sum())
    fn = int(((preds==0) & (labels==1)).sum())
    tn = int(((preds==0) & (labels==0)).sum())
    fp = int(((preds==1) & (labels==0)).sum())
    return {
        'auc'        : round(auc, 6),
        'f1'         : round(f1, 6),
        'loss'       : round(total_loss / max(n_batches, 1), 6),
        'sensitivity': round(tp / max(tp + fn, 1), 4),
        'specificity': round(tn / max(tn + fp, 1), 4),
    }


def train(model, train_loader, val_loader, test_loader,
          device, label, epochs, lr, patience, ckpt_path):

    optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr, weight_decay=1e-3,
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=epochs, eta_min=1e-6
    )
    best_val_auc, patience_ctr = 0.0, 0

    print(f'\n{"\u2550"*65}')
    print(f' {label}')
    print(f'{"\u2550"*65}')
    print(f' {"Ep":>3}  {"TrLoss":>8}  {"VaLoss":>8}  '
          f'{"VaAUC":>7}  {"VaF1":>7}  {"Secs":>6}')
    print(f'{"\u2500"*65}')

    for ep in range(1, epochs + 1):
        t0         = time.time()
        train_loss = train_one_epoch(model, train_loader, optimizer, device)
        val_m      = evaluate(model, val_loader, device)
        scheduler.step()

        flag = ''
        if val_m['auc'] > best_val_auc:
            best_val_auc = val_m['auc']
            patience_ctr = 0
            flag         = '\u2713'
            torch.save({
                'model_state': model.state_dict(),
                'epoch'      : ep,
                'val_auc'    : best_val_auc,
                'label'      : label,
            }, ckpt_path)
        else:
            patience_ctr += 1

        overfit_flag = ' \u26a0 overfit' if val_m['loss'] > train_loss * 2.5 else ''

        print(f' {ep:>3}  {train_loss:>8.4f}  {val_m["loss"]:>8.4f}  '
              f'{val_m["auc"]:>7.4f}  {val_m["f1"]:>7.4f}  '
              f'{int(time.time()-t0):>5}s  {flag}{overfit_flag}')

        if patience_ctr >= patience:
            print(f'\n Early stop \u2014 epoch {ep} '
                  f'(no improvement for {patience} epochs)')
            break

        torch.cuda.empty_cache()
        gc.collect()

    if Path(ckpt_path).exists():
        ck = torch.load(ckpt_path, weights_only=False)
        model.load_state_dict(ck['model_state'])

    test_m = evaluate(model, test_loader, device)

    print(f'{"\u2500"*65}')
    print(f' Best val AUC  : {best_val_auc:.4f}')
    print(f' Test AUC      : {test_m["auc"]:.4f}')
    print(f' Test F1       : {test_m["f1"]:.4f}')
    print(f' Sensitivity   : {test_m["sensitivity"]:.4f}')
    print(f' Specificity   : {test_m["specificity"]:.4f}')
    print(f' Overfit gap   : {test_m["auc"] - best_val_auc:+.4f}')
    print(f'{"\u2550"*65}')

    return {
        'val_auc'    : best_val_auc,
        'test_auc'   : test_m['auc'],
        'f1'         : test_m['f1'],
        'sensitivity': test_m['sensitivity'],
        'specificity': test_m['specificity'],
        'gap'        : round(test_m['auc'] - best_val_auc, 6),
    }

In [ ]:
# ── Build model ──────────────────────────────────────────────────────────
model = QuantaPathV2(
    use_vqc    = cfg['use_vqc'],
    n_qubits   = cfg['n_qubits'],
    vqc_layers = cfg['n_layers'],
    in_dim     = cfg['in_dim'],
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model built: {n_params:,} trainable parameters')
print(f'VQC active : {cfg["use_vqc"]}')

# ── Train ────────────────────────────────────────────────────────────
ckpt_path = str(CKPT_DIR / f'{EXPERIMENT}_best.pth')

result = train(
    model        = model,
    train_loader = train_loader,
    val_loader   = val_loader,
    test_loader  = test_loader,
    device       = DEVICE,
    label        = EXPERIMENT,
    epochs       = cfg['epochs'],
    lr           = cfg['lr'],
    patience     = cfg['patience'],
    ckpt_path    = ckpt_path,
)

# ── Save result JSON ─────────────────────────────────────────────────
result_data = {
    'experiment' : EXPERIMENT,
    'description': cfg['description'],
    'config'     : {
        'use_vqc'    : cfg['use_vqc'],
        'n_qubits'   : cfg['n_qubits'],
        'n_layers'   : cfg['n_layers'],
        'in_dim'     : cfg['in_dim'],
        'epochs'     : cfg['epochs'],
        'lr'         : cfg['lr'],
        'max_patches': 3000,
    },
    'results'    : result,
}

result_path = OUT_DIR / f'{EXPERIMENT}_result.json'
with open(result_path, 'w') as f:
    json.dump(result_data, f, indent=2)

print(f'\nResult saved : {result_path}')
print(f'Checkpoint   : {ckpt_path}')

In [ ]:
# ══════════════════════════════════════════════════════════
# RUN THIS CELL before your 12-hour Kaggle session ends
# It copies your checkpoint and result to /kaggle/working/
# Then download them from the Kaggle output panel on the right
# ══════════════════════════════════════════════════════════

import shutil

files_to_download = [
    (CKPT_DIR / f'{EXPERIMENT}_best.pth',    'checkpoint'),
    (OUT_DIR  / f'{EXPERIMENT}_result.json', 'result'),
]

print('Files ready to download:')
print()
for src, label in files_to_download:
    if src.exists():
        dest    = Path('/kaggle/working') / src.name
        shutil.copy(src, dest)
        size_mb = src.stat().st_size / 1e6
        print(f'  \u2705 {src.name}  ({size_mb:.1f} MB)  \u2190 {label}')
    else:
        print(f'  \u274c {src.name}  NOT FOUND \u2014 training may not have run')

print()
print('How to download:')
print('  1. Look at the right panel in Kaggle \u2192 Output')
print('  2. Click the file name \u2192 Download')
print('  3. Save to your laptop immediately')
print()
print('After downloading, your experiment is safely saved.')
print('You can stop the Kaggle session.')